In [0]:
%sql
-- 1. Overall KPI Summary — total net revenue, orders, active customers, avg order value
SELECT 
    ROUND(SUM(net_revenue), 2)                    AS total_net_revenue,
    SUM(total_orders)                              AS total_orders,
    MAX(active_customers)                          AS peak_active_customers,
    ROUND(SUM(net_revenue) / SUM(total_orders), 2) AS overall_avg_order_value
FROM tpch.gold.vw_annual_sales_kpis

In [0]:
%sql
-- 2. Monthly Revenue Trend Across All Years
SELECT 
    DATE_TRUNC('month', order_date) AS month,
    ROUND(SUM(net_price), 2)        AS monthly_revenue
FROM tpch.gold.fact_sales
GROUP BY DATE_TRUNC('month', order_date)
ORDER BY month

In [0]:
%sql
-- 3. Annual Revenue with Year-over-Year Change
SELECT 
    sales_year,
    ROUND(net_revenue, 2)                                                          AS net_revenue,
    ROUND(LAG(net_revenue) OVER (ORDER BY sales_year), 2)                          AS prev_year_revenue,
    ROUND(net_revenue - LAG(net_revenue) OVER (ORDER BY sales_year), 2)            AS yoy_change,
    ROUND(
        (net_revenue - LAG(net_revenue) OVER (ORDER BY sales_year)) 
        / LAG(net_revenue) OVER (ORDER BY sales_year) * 100, 2
    )                                                                              AS yoy_pct_change
FROM tpch.gold.vw_annual_sales_kpis
ORDER BY sales_year

In [0]:
%sql
-- 4. Revenue by Customer Market Segment
SELECT 
    c.market_segment,
    ROUND(SUM(f.net_price), 2)                      AS net_revenue,
    COUNT(DISTINCT f.customer_key)                   AS customers,
    ROUND(SUM(f.net_price) / COUNT(DISTINCT f.customer_key), 2) AS revenue_per_customer
FROM tpch.gold.fact_sales f
JOIN tpch.gold.dim_customer c ON f.customer_key = c.customer_key
GROUP BY c.market_segment
ORDER BY net_revenue DESC

In [0]:
%sql
-- 5. Top 10 Customers by Net Revenue
SELECT 
    c.customer_key,
    c.customer_name,
    c.market_segment,
    c.nation_name,
    ROUND(SUM(f.net_price), 2)    AS net_revenue,
    COUNT(DISTINCT f.order_key)   AS orders
FROM tpch.gold.fact_sales f
JOIN tpch.gold.dim_customer c ON f.customer_key = c.customer_key
GROUP BY c.customer_key, c.customer_name, c.market_segment, c.nation_name
ORDER BY net_revenue DESC
LIMIT 10

In [0]:
%sql
-- 6. Top 10 Customers as Share of Total Revenue
WITH top_customers AS (
    SELECT 
        c.customer_key,
        c.customer_name,
        SUM(f.net_price) AS net_revenue
    FROM tpch.gold.fact_sales f
    JOIN tpch.gold.dim_customer c ON f.customer_key = c.customer_key
    GROUP BY c.customer_key, c.customer_name
    ORDER BY net_revenue DESC
    LIMIT 10
),
total_revenue AS (
    SELECT SUM(net_price) AS grand_total FROM tpch.gold.fact_sales
)
SELECT 
    tc.customer_key,
    tc.customer_name,
    ROUND(tc.net_revenue, 2)                               AS net_revenue,
    ROUND(tc.net_revenue / tr.grand_total * 100, 4)         AS pct_of_total_revenue,
    ROUND(tr.grand_total, 2)                                AS total_revenue
FROM top_customers tc
CROSS JOIN total_revenue tr
ORDER BY tc.net_revenue DESC

In [0]:
%sql
-- 7. Top 10 Products by Net Revenue
SELECT 
    p.product_key,
    p.product_name,
    p.manufacturer,
    p.brand,
    ROUND(SUM(f.net_price), 2) AS net_revenue,
    SUM(f.quantity)             AS total_quantity
FROM tpch.gold.fact_sales f
JOIN tpch.gold.dim_product p ON f.product_key = p.product_key
GROUP BY p.product_key, p.product_name, p.manufacturer, p.brand
ORDER BY net_revenue DESC
LIMIT 10

In [0]:
%sql
-- 8. Bottom 10 Products by Net Revenue
SELECT 
    p.product_key,
    p.product_name,
    p.manufacturer,
    p.brand,
    ROUND(SUM(f.net_price), 2) AS net_revenue,
    SUM(f.quantity)             AS total_quantity
FROM tpch.gold.fact_sales f
JOIN tpch.gold.dim_product p ON f.product_key = p.product_key
GROUP BY p.product_key, p.product_name, p.manufacturer, p.brand
ORDER BY net_revenue ASC
LIMIT 10

In [0]:
%sql
-- 9. Revenue by Product Type
SELECT 
    p.product_type,
    ROUND(SUM(f.net_price), 2)   AS net_revenue,
    SUM(f.quantity)              AS total_quantity,
    COUNT(DISTINCT p.product_key) AS products
FROM tpch.gold.fact_sales f
JOIN tpch.gold.dim_product p ON f.product_key = p.product_key
GROUP BY p.product_type
ORDER BY net_revenue DESC

In [0]:
%sql
-- 10. Top 10 Suppliers by Net Revenue
SELECT 
    s.supplier_key,
    s.supplier_name,
    s.nation_name,
    s.region_name,
    ROUND(SUM(f.net_price), 2)           AS net_revenue,
    COUNT(DISTINCT f.product_key)        AS products_supplied
FROM tpch.gold.fact_sales f
JOIN tpch.gold.dim_supplier s ON f.supplier_key = s.supplier_key
GROUP BY s.supplier_key, s.supplier_name, s.nation_name, s.region_name
ORDER BY net_revenue DESC
LIMIT 10

In [0]:
%sql
-- 11. Top 10 Suppliers as Share of Total Revenue
WITH top_suppliers AS (
    SELECT 
        s.supplier_key,
        s.supplier_name,
        SUM(f.net_price) AS net_revenue
    FROM tpch.gold.fact_sales f
    JOIN tpch.gold.dim_supplier s ON f.supplier_key = s.supplier_key
    GROUP BY s.supplier_key, s.supplier_name
    ORDER BY net_revenue DESC
    LIMIT 10
),
total_revenue AS (
    SELECT SUM(net_price) AS grand_total FROM tpch.gold.fact_sales
)
SELECT 
    ts.supplier_key,
    ts.supplier_name,
    ROUND(ts.net_revenue, 2)                        AS net_revenue,
    ROUND(ts.net_revenue / tr.grand_total * 100, 4)  AS pct_of_total_revenue,
    ROUND(tr.grand_total, 2)                          AS total_revenue
FROM top_suppliers ts
CROSS JOIN total_revenue tr
ORDER BY ts.net_revenue DESC

In [0]:
%sql
-- 12. Revenue by Customer Region
SELECT 
    c.region_name,
    ROUND(SUM(f.net_price), 2)        AS net_revenue,
    COUNT(DISTINCT f.customer_key)     AS customers,
    COUNT(DISTINCT f.order_key)        AS orders
FROM tpch.gold.fact_sales f
JOIN tpch.gold.dim_customer c ON f.customer_key = c.customer_key
GROUP BY c.region_name
ORDER BY net_revenue DESC

In [0]:
%sql
-- 13. Revenue by Customer Nation (with region for context)
SELECT 
    c.region_name,
    c.nation_name,
    ROUND(SUM(f.net_price), 2)        AS net_revenue,
    COUNT(DISTINCT f.customer_key)     AS customers
FROM tpch.gold.fact_sales f
JOIN tpch.gold.dim_customer c ON f.customer_key = c.customer_key
GROUP BY c.region_name, c.nation_name
ORDER BY c.region_name, net_revenue DESC

In [0]:
%sql
-- 14. Revenue and Volume Split by Return Flag
SELECT 
    f.return_flag,
    ROUND(SUM(f.net_price), 2)   AS net_revenue,
    SUM(f.quantity)               AS total_quantity,
    COUNT(*)                       AS line_count,
    ROUND(SUM(f.net_price) / SUM(SUM(f.net_price)) OVER () * 100, 2) AS pct_of_revenue
FROM tpch.gold.fact_sales f
GROUP BY f.return_flag
ORDER BY net_revenue DESC

In [0]:
%sql
-- 15. Avg Discount by Market Segment
SELECT 
    c.market_segment,
    ROUND(AVG(f.discount) * 100, 2)   AS avg_discount_pct,
    ROUND(MIN(f.discount) * 100, 2)   AS min_discount_pct,
    ROUND(MAX(f.discount) * 100, 2)   AS max_discount_pct,
    COUNT(*)                           AS line_count
FROM tpch.gold.fact_sales f
JOIN tpch.gold.dim_customer c ON f.customer_key = c.customer_key
GROUP BY c.market_segment
ORDER BY avg_discount_pct DESC

In [0]:
%sql
-- 16. Avg Ship-to-Receipt Days by Shipping Mode
SELECT 
    f.ship_mode,
    ROUND(AVG(DATEDIFF(f.receipt_date, f.ship_date)), 1) AS avg_days,
    ROUND(MIN(DATEDIFF(f.receipt_date, f.ship_date)), 0)  AS min_days,
    ROUND(MAX(DATEDIFF(f.receipt_date, f.ship_date)), 0)  AS max_days,
    COUNT(*)                                                AS line_count
FROM tpch.gold.fact_sales f
GROUP BY f.ship_mode
ORDER BY avg_days DESC

In [0]:
%sql
-- 17. Order Count and Revenue by Order Priority
SELECT 
    f.order_priority,
    COUNT(DISTINCT f.order_key)                           AS order_count,
    ROUND(SUM(f.net_price), 2)                            AS net_revenue,
    ROUND(SUM(f.net_price) / COUNT(DISTINCT f.order_key), 2) AS avg_revenue_per_order
FROM tpch.gold.fact_sales f
GROUP BY f.order_priority
ORDER BY net_revenue DESC

In [0]:
%sql
-- 18. Monthly Breakdown of Order Status
SELECT 
    DATE_TRUNC('month', order_date) AS month,
    order_status,
    COUNT(DISTINCT order_key)       AS order_count,
    ROUND(SUM(net_price), 2)        AS net_revenue
FROM tpch.gold.fact_sales
GROUP BY DATE_TRUNC('month', order_date), order_status
ORDER BY month, order_status

In [0]:
%sql
-- 19. Product Revenue Ranking Within Each Market Segment (top 10 per segment)
WITH product_segment_revenue AS (
    SELECT 
        c.market_segment,
        p.product_key,
        p.product_name,
        ROUND(SUM(f.net_price), 2) AS net_revenue
    FROM tpch.gold.fact_sales f
    JOIN tpch.gold.dim_customer c ON f.customer_key = c.customer_key
    JOIN tpch.gold.dim_product p ON f.product_key = p.product_key
    GROUP BY c.market_segment, p.product_key, p.product_name
),
ranked AS (
    SELECT 
        market_segment,
        product_key,
        product_name,
        net_revenue,
        ROW_NUMBER() OVER (PARTITION BY market_segment ORDER BY net_revenue DESC) AS rank_in_segment
    FROM product_segment_revenue
)
SELECT 
    market_segment,
    rank_in_segment,
    product_key,
    product_name,
    net_revenue
FROM ranked
WHERE rank_in_segment <= 10
ORDER BY market_segment, rank_in_segment

In [0]:
%sql
-- 20. Revenue Comparison: Customer Geography vs Supplier Geography by Region
WITH customer_revenue AS (
    SELECT 
        c.region_name,
        SUM(f.net_price) AS customer_revenue
    FROM tpch.gold.fact_sales f
    JOIN tpch.gold.dim_customer c ON f.customer_key = c.customer_key
    GROUP BY c.region_name
),
supplier_revenue AS (
    SELECT 
        s.region_name,
        SUM(f.net_price) AS supplier_revenue
    FROM tpch.gold.fact_sales f
    JOIN tpch.gold.dim_supplier s ON f.supplier_key = s.supplier_key
    GROUP BY s.region_name
)
SELECT 
    COALESCE(cr.region_name, sr.region_name) AS region,
    ROUND(COALESCE(cr.customer_revenue, 0), 2)  AS customer_revenue,
    ROUND(COALESCE(sr.supplier_revenue, 0), 2)  AS supplier_revenue,
    ROUND(COALESCE(cr.customer_revenue, 0) - COALESCE(sr.supplier_revenue, 0), 2) AS revenue_difference
FROM customer_revenue cr
FULL OUTER JOIN supplier_revenue sr ON cr.region_name = sr.region_name
ORDER BY region